# Self-Pruning Neural Network on CIFAR-10

The idea here is to build a network that figures out which of its own weights are useless — and learns to zero them out *while* training, not after.

Each linear layer gets learnable "gate" scores attached to its weights. During the forward pass, gates (passed through sigmoid) get multiplied onto the weights. If the optimizer decides a gate isn't worth keeping, it pushes it toward zero — effectively removing that connection.

We control how aggressively this happens using a sparsity penalty (λ) in the loss.

**Outline:**
1. `PrunableLinear` — custom linear layer with per-weight gate scores
2. Sparsity loss — L1 penalty on gate values
3. Training on CIFAR-10 with three λ values
4. Accuracy vs sparsity tradeoff analysis

In [ ]:
# install dependencies if not already present
import subprocess, sys

pkgs = ["torch", "torchvision", "matplotlib"]
subprocess.check_call([sys.executable, "-m", "pip", "install", *pkgs, "--quiet"])
print("all good")

In [ ]:
import math
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

import torchvision
import torchvision.transforms as transforms

import matplotlib.pyplot as plt
import numpy as np

print(f"torch version: {torch.__version__}")
print(f"cuda available: {torch.cuda.is_available()}")